In [2]:
import copy
import torch
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import pytorch_lightning as pl
from pytorch_forecasting.data import GroupNormalizer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_forecasting.data.examples import get_stallion_data
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
warnings.filterwarnings("ignore")
from tqdm import tqdm
import matplotlib.pyplot as plt
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from datetime import timedelta
import global_functions as gl

In [8]:
date_after = '2026-04-01'

In [9]:
sql_inc = """
SELECT 
[vbeln] as [talep_no],
[fkdat] as [tarihfatura],
[kunrg] as [kunnr],
[matnr] as [mamulno],
[fkimgton] as [miktar_ton],
[fkimg] as [adet],
[rafsatfyt],
[epdkkatpay] as [epdk],
[otv] as [otv],
[perlistfyt],
[hesmrj],
[hesnav],
[baydgtpay],
[aygpay],
[WADAT_IST] as [fiilimalcıkıs]
FROM [sqldb-aygaz-prod-001].[ods].[zaygbw_tup_ortfy]
WHERE matnr = 000000000006040012
AND fkdat > ?
"""

fatura = gl.read_from_sql_multi(sql_inc, "Managed_Instance_BigDataUsr", params=(date_after,))
fatura["tarihfatura"] = pd.to_datetime(fatura["tarihfatura"], errors="coerce")


In [10]:
fatura.drop(fatura[fatura["hesnav"]==0].index, inplace=True)
fatura = fatura[fatura["fiilimalcıkıs"]> '2023-01-01']
fatura.sort_values(by=["fiilimalcıkıs"], ascending=True, inplace=True)
fatura["mamulno"]=fatura["mamulno"].astype(int)
fatura["kunnr"]=(fatura["kunnr"].astype(int)).astype(str)

In [13]:
sql_query = f"""
SELECT 
      [ERP_RETAILER_CODE]
      ,[RETAILER_NAME]
      ,[PROVINCE_ID]
      ,[TOWN_ID]
      ,COMPANY_REGION_NO
  FROM [ESSNET].[dbo].[RETAILER]
    """
bayiler = gl.read_from_sql_multi(sql_query, 'ESSNET')

ProgrammingError: (pyodbc.ProgrammingError) ('42000', "[42000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Logon failed for login 'EssNetReportUsr' due to trigger execution. (17892) (SQLDriverConnect)")
(Background on this error at: https://sqlalche.me/e/20/f405)